In [65]:
import pandas as pd
from scipy.stats import iqr

In [66]:
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)

In [67]:
df = pd.read_csv('../data/operations.csv')
df = df.convert_dtypes()

In [68]:
df['mortality'] = df['inhosp_death_time'].notna().astype(int)

In [69]:
df = df.drop(columns=['race']) #Single unique value → no predictive power

In [70]:
df = df.drop(columns=['cpbon_time','cpboff_time',
                      'icuin_time','icuout_time','case_id','inhosp_death_time','allcause_death_time',]) 
#Columns with extremely high missingness (>80%) would require heavy imputation, which could introduce bias and reduce model reliability.

In [71]:
df = df.sort_values(['subject_id','admission_time','opstart_time'])

In [79]:
# df['admission_count'] = df.groupby('subject_id')['hadm_id'].transform(lambda x: pd.factorize(x)[0] + 1)
# df['operation_count'] = df.groupby('subject_id').cumcount() + 1

In [72]:
df = df[df['opdate'] == 0] #filtered to the first surgical event to standardize patient timelines and avoid bias from repeated procedures.

In [73]:
mask = df.groupby('subject_id')['hadm_id'].transform('count') == 1 #To simplify the problem and avoid intra-patient correlation,
df = df[mask] #restricted to one encounter per patient for this initial model.

In [74]:
df = df.dropna(how='any', subset=['opend_time','opstart_time','anstart_time','anend_time']) # Data missing in these columns are less than 1 percent

In [75]:
cols = ['admission_time','orin_time','anstart_time','opstart_time','anend_time','opend_time','orout_time','discharge_time']
for col in cols:
  df[col] = pd.to_timedelta(df[col], unit='m')

In [76]:
outlier_indices = df[df['opstart_time'] >= pd.Timedelta(days=9)].index #Exploratory analysis showed this value was significantly beyond
df = df.drop(outlier_indices) #the typical distribution, likely due to data entry error or exceptional case.likely due to data entry error or exceptional case.

In [77]:
# df[['height','weight','asa']].describe()
df['asa']=df['asa'].fillna(df['asa'].mode()[0]) #imputing mode as it is an ordinal column
df['height'] = df['height'].fillna(df['height'].median()) #imputing with median since the data is right skewed
df['weight'] = df['weight'].fillna(df['weight'].median()) #imputing with median since the data is right skewed

In [78]:
def winsorization(col):
    _iqr = iqr(df[col]) 
    uplmt = df[col].quantile(0.75) + 1.5 * _iqr 
    lolmt = df[col].quantile(0.25) - 1.5 * _iqr 
    df[col] = df[col].astype('float64').round(2) 
    df[col] = df[col].clip(upper = uplmt, lower=lolmt) 
    print("Done") 
winsorization('height') 
winsorization('weight')

Done
Done


In [79]:
df['surgery_duration'] = df['opend_time'] - df['opstart_time']
df['anesthesia_duration'] = df['anend_time'] - df['anstart_time']
df['preop_wait_time'] = df['opstart_time'] - df['admission_time']
df['postop_stay'] = df['discharge_time'] - df['opend_time']

In [80]:
time_cols = ['surgery_duration','anesthesia_duration','preop_wait_time','orin_time','orout_time','opstart_time','opend_time',
    'admission_time','anstart_time','anend_time']
for col in time_cols:
    df[col] = df[col].dt.total_seconds()

In [81]:
df['bmi'] = df['weight'] / (df['height']/100)**2

In [82]:
df['bmi_category'] = pd.cut(df['bmi'],
                           bins=[0,18.5,25,30,100],
                           labels=['underweight','normal','overweight','obese'])

In [83]:
df['high_risk'] = (df['asa'] >= 3).astype(int)

In [84]:
df['age_group'] = pd.cut(df['age'],
                        bins=[0,30,50,70,100],
                        labels=['young','mid','senior','elderly'])

In [85]:
df['risk_x_duration'] = df['asa'] * df['surgery_duration']

In [86]:
#Data leakage occurs when the model is trained on information that would not be available at prediction time. In this dataset, 
# features like postop_stay and discharge_time include post-operative information, which is influenced by the patient’s outcome. 
# Including them would artificially inflate model performance, so I removed them.

cols_to_drop = ['postop_stay','discharge_time']
df = df.drop(columns=cols_to_drop)

In [87]:
df.shape

(11050, 29)

In [88]:
df.to_csv('../data/processed/clnopr.csv')